In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [2]:
words = open('names.txt').read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [3]:
# build the vocabulary of characters and mapping to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [4]:
# build the dataset

block_size= 3
X, Y = [], []
for w in words:

    # print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        # print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

In [ ]:
C = torch.randn((27, 2))
emb = C[X]
W1 = torch.randn((6, 100))
b1 = torch.randn(100)
emb.view(32, 6)
# torch.cat(torch.unbind(emb, 1), 1) not recommend

In [ ]:
h = torch.tanh(emb.view(emb.shape[0], 6) @ W1 + b1)
h.shape

In [ ]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)
logits = h @ W2 + b2
logits.shape

In [ ]:
prob = (logits.exp()) / (logits.exp()).sum(1, keepdim=True)
prob.shape

In [ ]:
loss = -prob[torch.arange(32), Y].log().mean()
loss

In [ ]:
# -------------- now make it respectable ------------------------

In [5]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [6]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [9]:
for p in parameters:
    p.requires_grad = True

for _ in range(100):

    # minibatch construct
    ix = torch.randint(0, X.shape[0], (32,))

    # forward pass
    emb = C[X[ix]] # (32, 3, 2)
    h = torch.tanh(emb.view(emb.shape[0], 6) @ W1 + b1) # (32, 100)
    logits = h @ W2 + b2 # (32, 27)
    loss = F.cross_entropy(logits, Y[ix])
    print(loss.item())

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad

3.006834030151367
2.919316291809082
2.923149585723877
3.0488343238830566
3.086951732635498
3.4715707302093506
3.0146636962890625
3.609971284866333
3.1071553230285645
3.038409471511841
3.3451316356658936
2.601088285446167
2.6567893028259277
3.268082857131958
3.246326446533203
2.8844339847564697
2.779630422592163
2.823143720626831
2.869379997253418
3.5357513427734375
2.695465087890625
3.2691519260406494
2.993675708770752
3.3540384769439697
3.1068081855773926
2.735928773880005
2.8896477222442627
3.3242063522338867
2.543924570083618
3.539085626602173
2.768355369567871
2.957537889480591
4.078354835510254
2.892240047454834
3.108536720275879
2.7826449871063232
3.5387730598449707
3.038825750350952
2.9999258518218994
3.086031675338745
3.254317283630371
2.6370251178741455
2.503591537475586
2.897984266281128
2.527111530303955
2.7001826763153076
2.6869864463806152
3.177014112472534
2.866489887237549
3.1799070835113525
2.9148340225219727
2.8594565391540527
3.3060364723205566
3.3007261753082275
3.44